# Human RefSeq transcript coding-status study

**Question:** In a pinned human RefSeq annotation snapshot, how well can nucleotide sequence features distinguish transcripts annotated with a CDS from explicitly annotated noncoding transcripts?

This focused, preliminary study uses versioned human RefSeq NM_/NR_ transcript records on GRCh38.p14. It excludes predicted XM_/XR_ models, pseudogenes, unresolved labels, FLIP2, and variant information. Labels come from NCBI GFF3 feature evidence and must agree with the RefSeq accession prefix.

The notebook audits the full source files, then uses a reproducible class-balanced sample of at most 3,000 records per label for modeling. GeneID, exact duplicates, and MMseqs2 similarity clusters are joined into connected groups before splitting. The primary split holds out complete groups. It compares a majority baseline with two logistic-regression models and reports held-out results and similarity-threshold sensitivity.


## Data, labels, and limits

The source is the fixed NCBI annotation release GCF_000001405.40-RS_2025_08 for GRCh38.p14, with accession-versioned RNA FASTA, genomic GFF3, assembly report, and NCBI MD5 checksums. Files are verified against NCBI MD5 values and independently recorded with SHA-256.

A transcript is labeled coding only when its GFF3 transcript relationship is linked to a CDS feature on an assembled molecule listed in the assembly report. It is labeled noncoding only when it has no CDS there and its transcript feature or biotype explicitly identifies a noncoding RNA. The included assembled molecules are chromosomes 1–22, X, Y, and mitochondrion; alternate placements are excluded. Pseudogenes, conflicting or unresolved GFF evidence, missing or conflicting GeneIDs, invalid sequences, and NM_/NR_ prefix disagreements are excluded from modeling and counted in the audit. The accession prefix is a source-consistency check, not the label-generating rule.

This measures agreement with one NCBI annotation snapshot. It is not an experimental assay of translation, expression, or biological function. Threshold sensitivity uses separate grouped test splits and is descriptive; it does not select the primary model. Sampling is uniform over transcripts, so genes with many isoforms can contribute more records; point estimates are transcript-weighted.


In [ ]:
import gzip, hashlib, json, os, platform, re, shutil, subprocess, tarfile, tempfile, urllib.request, zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
ASSEMBLY = "GCF_000001405.40"
RELEASE = "GCF_000001405.40-RS_2025_08"
ASSEMBLY_NAME = "GRCh38.p14"
RELEASE_URL = "https://ftp.ncbi.nlm.nih.gov/genomes/all/annotation_releases/9606/GCF_000001405.40-RS_2025_08/"
REPORT_URL = "https://www.ncbi.nlm.nih.gov/refseq/annotation_euk/Homo_sapiens/GCF_000001405.40-RS_2025_08/"
RNA = "GCF_000001405.40_GRCh38.p14_rna.fna.gz"
GFF = "GCF_000001405.40_GRCh38.p14_genomic.gff.gz"
ASSEMBLY_REPORT = "GCF_000001405.40_GRCh38.p14_assembly_report.txt"
MMSEQS_ARCHIVE = "mmseqs-linux-avx2.tar.gz"
INPUT_BUNDLE = "refseq_human_transcript_inputs.zip"
MMSEQS_SHA256 = "610abd77e0fe4feeb63c407c129caa3726e502256e54565cc9c02f0233800dc9"
MMSEQS_BINARY_SHA256 = "98cc37a85e51a891733d9e40dc2ae3d02fa7e02cbddaabb192c8f8e6fba8ad32"
MD5CHECKSUMS_SHA256 = "a6ad1c10ef1b48ee83c24742fa3a092a244b01615a174d470012d1ae09241e56"
SOURCE_SHA256 = {
    RNA: "e214c2e0a00a5e8726487b611fc5afb166ca776b11106f1211ee7d14c095d9e2",
    GFF: "4920f0eae7e2197c50b67a201e06d657387137b49dd60f474b4f1d5b29334051",
    ASSEMBLY_REPORT: "64318ddff470b69b261a667d813210044f60d4ce654253a547db80ff73638d38",
}
EXPANDED_SOURCE_SHA256 = {
    RNA: "624395703c396969d12374c3211cd9cf73fcda04106e819032740f15cc9b3fe5",
    GFF: "cb83677489562c165a046c8833813e81d36f0b703d2ebbb2e3c6adb84a5c0780",
}
MIN_LENGTH = 20
MAX_PER_CLASS = 3000
IDENTITIES = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
COVERAGE = 0.80
WORK = Path("refseq_transcript_work")
OUTPUT = Path("outputs/transcript_coding_status")
WORK.mkdir(exist_ok=True, parents=True)
OUTPUT.mkdir(exist_ok=True, parents=True)
np.random.seed(SEED)
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "scikit_learn": sklearn.__version__, "release": RELEASE,
       "retrieved_utc": datetime.now(timezone.utc).isoformat()})


## 1. Pin and verify data and software

For a Kaggle rerun with Internet disabled, attach the private input dataset containing the NCBI bundle and pinned MMseqs2 distribution. Kaggle may expose the ZIP contents in expanded form, so the notebook supports both the original compressed source files and Kaggle's extracted FASTA/GFF/MMseqs2 files. It pins the NCBI checksum-list digest and original or expanded source SHA-256 values, plus the MMseqs2 archive or executable digest. When original compressed bytes are available, it also verifies NCBI MD5 values; Kaggle-expanded files are checked against pinned content SHA-256 values. It does not substitute an unverified system binary. No raw sequences are written to the Flyrank repository.


In [ ]:
ROOTS = [Path("/kaggle/input"), Path.cwd(), WORK]
if Path("/content").exists():
    ROOTS.append(Path("/content"))

for root in ROOTS:
    if not root.exists():
        continue
    try:
        bundles = list(root.rglob(INPUT_BUNDLE))
    except (PermissionError, OSError):
        bundles = []
    for bundle in bundles:
        extracted = WORK / "attached_input_bundle"
        base = extracted.resolve()
        extracted.mkdir(exist_ok=True, parents=True)
        with zipfile.ZipFile(bundle) as archive:
            for name in archive.namelist():
                target = (extracted / name).resolve()
                if base not in target.parents and target != base:
                    raise RuntimeError("Unsafe path in attached input bundle")
            archive.extractall(extracted)

def find_input(name):
    candidates = [name]
    if name.endswith(".gz"):
        candidates.append(name[:-3])
    for root in ROOTS:
        if root.exists():
            for candidate in candidates:
                try:
                    found = next(root.rglob(candidate), None)
                except (PermissionError, OSError):
                    found = None
                if found and found.is_file():
                    return found
    return None

def digest(path, algorithm):
    h = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checksum_file = find_input("md5checksums.txt") or WORK / "md5checksums.txt"
if not checksum_file.exists():
    urllib.request.urlretrieve(RELEASE_URL + "md5checksums.txt", checksum_file)
if digest(checksum_file, "sha256") != MD5CHECKSUMS_SHA256:
    raise RuntimeError("NCBI checksum-list SHA-256 does not match the pinned release snapshot")
ncbi_md5 = {}
for line in checksum_file.read_text(errors="replace").splitlines():
    fields = line.split()
    if len(fields) >= 2:
        ncbi_md5[Path(fields[-1].lstrip("*")).name] = fields[0].lower()

sources = []
for name in [RNA, GFF, ASSEMBLY_REPORT]:
    target = WORK / name
    attached = find_input(name)
    if attached and attached.resolve() != target.resolve():
        shutil.copy2(attached, target)
    elif not target.exists():
        urllib.request.urlretrieve(RELEASE_URL + name, target)
    with open(target, "rb") as stream:
        gzip_encoded = stream.read(2) == b"\x1f\x8b"
    if name not in ncbi_md5:
        raise RuntimeError(f"Pinned NCBI checksum list does not contain {name}")
    sha256 = digest(target, "sha256")
    if name in [RNA, GFF] and not gzip_encoded:
        expected = EXPANDED_SOURCE_SHA256[name]
        if sha256 != expected:
            raise RuntimeError(f"Expanded NCBI source SHA-256 check failed for {name}")
        representation = "expanded_by_Kaggle"
        md5_verified = False
    else:
        if digest(target, "md5") != ncbi_md5[name] or sha256 != SOURCE_SHA256[name]:
            raise RuntimeError(f"NCBI compressed/original source checksum check failed for {name}")
        representation = "original_source_bytes"
        md5_verified = True
    sources.append({"name":name,"bytes":target.stat().st_size,"representation":representation,
                    "ncbi_md5_expected":ncbi_md5[name],"ncbi_md5_verified":md5_verified,
                    "sha256":sha256,"sha256_verified":True,"url":RELEASE_URL+name})

assembly_text = (WORK / ASSEMBLY_REPORT).read_text(errors="replace")
if ASSEMBLY not in assembly_text or ASSEMBLY_NAME not in assembly_text:
    raise RuntimeError("Assembly report did not confirm the expected GRCh38.p14 accession")
mmseqs_archive = find_input(MMSEQS_ARCHIVE)
mmseqs_extract = None
if mmseqs_archive is not None and digest(mmseqs_archive, "sha256") == MMSEQS_SHA256:
    mmseqs_extract = tempfile.TemporaryDirectory(prefix="mmseqs_verified_", dir=WORK)
    extracted = Path(mmseqs_extract.name)
    base = extracted.resolve()
    with tarfile.open(mmseqs_archive, "r:gz") as archive:
        for member in archive.getmembers():
            target = (extracted / member.name).resolve()
            if base not in target.parents and target != base:
                raise RuntimeError("Unsafe path in MMseqs2 archive")
            if not (member.isdir() or member.isfile()):
                raise RuntimeError("Unexpected link or special file in MMseqs2 archive")
        archive.extractall(extracted)
    binaries = [p for p in extracted.rglob("mmseqs") if p.is_file()]
    mmseqs_representation = "pinned_archive"
else:
    binaries = []
    for root in ROOTS:
        if root.exists():
            try:
                binaries.extend(p for p in root.rglob("mmseqs")
                                if p.is_file() and p.parent.name == "bin")
            except (PermissionError, OSError):
                pass
    mmseqs_representation = "expanded_by_Kaggle"
binaries = [p for p in binaries if digest(p, "sha256") == MMSEQS_BINARY_SHA256]
if not binaries:
    raise FileNotFoundError("Pinned MMseqs2 archive or its expanded binary is missing")
binary_source = Path(binaries[0])
if digest(binary_source, "sha256") != MMSEQS_BINARY_SHA256:
    raise RuntimeError("MMseqs2 executable SHA-256 does not match the pinned distribution")
verified_binary_dir = WORK / "verified_mmseqs_executable"
verified_binary_dir.mkdir(exist_ok=True, parents=True)
mmseqs_cmd = str(verified_binary_dir / "mmseqs")
if binary_source.resolve() != Path(mmseqs_cmd).resolve():
    shutil.copy2(binary_source, mmseqs_cmd)
if digest(mmseqs_cmd, "sha256") != MMSEQS_BINARY_SHA256:
    raise RuntimeError("Copied MMseqs2 executable SHA-256 does not match the pinned distribution")
os.chmod(mmseqs_cmd, os.stat(mmseqs_cmd).st_mode | 0o111)
mmseqs_version = subprocess.check_output([mmseqs_cmd, "version"], text=True).strip()
print(pd.DataFrame(sources)[["name","bytes","representation","ncbi_md5_verified","sha256"]].to_string(index=False))
print("Assembly report verified:", ASSEMBLY, ASSEMBLY_NAME)
print("MMseqs2:", mmseqs_version, "input representation:", mmseqs_representation,
      "binary SHA-256:", digest(mmseqs_cmd,"sha256"))


## 2. Parse GFF3 labels and versioned FASTA records

GFF3 rows are aggregated by exact accession.version across transcript and CDS features. A coding label requires CDS evidence. A noncoding label requires the absence of CDS plus an explicit noncoding RNA feature or biotype. Conflicts and unresolvable records remain visible in the audit but are excluded.


In [ ]:
ACCESSION_RE = re.compile(r"\b(?:NM|NR)_\d+\.\d+\b")
GENEID_RE = re.compile(r"(?:GeneID[:=]|gene_id=)(\d+)", re.IGNORECASE)
NONCODING = ("ncrna","noncoding","lncrna","lnc_rna","mirna","snrna","snorna","trna","rrna",
             "misc_rna","precursor_rna","antisense","pirna","scarna","vault_rna","telomerase_rna",
             "srp_rna","rnase_p_rna","rnase_mrp_rna","y_rna","guide_rna","ribozyme")

report_lines = assembly_text.splitlines()
report_header = next((line.lstrip("# ").split("\t") for line in report_lines
                      if line.startswith("# Sequence-Name\t")),None)
if report_header is None:
    raise RuntimeError("Assembly report sequence table is missing")
report_rows = [dict(zip(report_header,line.split("\t"))) for line in report_lines
               if line and not line.startswith("#") and len(line.split("\t"))==len(report_header)]
primary_seqids = {row["RefSeq-Accn"] for row in report_rows
                  if row.get("Sequence-Role")=="assembled-molecule" and row.get("RefSeq-Accn")}
if len(primary_seqids)<24:
    raise RuntimeError("Assembly report did not identify primary assembled molecules")

annotation = defaultdict(lambda: {"features":set(),"gene_ids":set(),"biotypes":set(),
                                  "has_cds":False,"pseudogene":False})
gff_rows_total=0
gff_rows_primary=0
gff_rows_outside_primary=0
def open_source_text(path):
    with open(path, "rb") as stream:
        gzip_encoded = stream.read(2) == b"\x1f\x8b"
    return gzip.open(path, "rt", errors="replace") if gzip_encoded else open(path, "rt", errors="replace")

with open_source_text(WORK / GFF) as stream:
    for line in stream:
        if not line or line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9:
            continue
        gff_rows_total+=1
        if fields[0] not in primary_seqids:
            gff_rows_outside_primary+=1
            continue
        gff_rows_primary+=1
        feature = fields[2].lower()
        attrs = {}
        for item in fields[8].split(";"):
            if "=" in item:
                key, value = item.split("=",1)
                attrs[unquote(key).lower()] = unquote(value)
        transcript_values=[]
        for key in ("transcript_id","id","parent"):
            transcript_values.extend(attrs.get(key,"").split(","))
        accessions=set()
        for value in transcript_values:
            value=value.strip()
            for prefix in ("rna-","transcript-"):
                if value.startswith(prefix): value=value[len(prefix):]
            match=ACCESSION_RE.fullmatch(value)
            if match: accessions.add(match.group(0))
        if not accessions:
            continue
        ids = set(GENEID_RE.findall(" ".join(attrs.values())))
        biotypes = {str(value).lower() for key,value in attrs.items()
                    if key in {"gene_biotype","transcript_biotype","gene_type","biotype","gbkey"}}
        is_pseudo = (
            "pseudogene" in feature
            or any("pseudogene" in value for value in biotypes)
            or attrs.get("pseudo", "").lower() == "true"
            or "pseudogene" in attrs.get("product", "").lower()
        )
        for accession in accessions:
            item = annotation[accession]
            item["features"].add(feature)
            item["gene_ids"].update(ids)
            item["biotypes"].update(biotypes)
            item["has_cds"] = item["has_cds"] or feature == "cds"
            item["pseudogene"] = item["pseudogene"] or is_pseudo

def label_from_gff(item):
    if not item:
        return "unmatched"
    if item["pseudogene"]:
        return "pseudogene"
    joined = " ".join(item["features"] | item["biotypes"]).lower()
    explicit_non = any(term in joined for term in NONCODING)
    if item["has_cds"] and explicit_non:
        return "ambiguous"
    if item["has_cds"]:
        return "coding"
    if explicit_non:
        return "noncoding"
    return "ambiguous"

def fasta_accession(header):
    match = ACCESSION_RE.search(header)
    return match.group(0) if match else None

rows, header, parts = [], None, []
def flush():
    if header is None:
        return
    accession = fasta_accession(header)
    if accession is None:
        return
    sequence = "".join(parts).upper().replace("U","T")
    rows.append({"accession":accession,"prefix":accession[:2],"sequence":sequence,
                 "length":len(sequence),"valid_bases":bool(re.fullmatch(r"[ACGTN]+",sequence)),
                 "header_gene_ids":sorted(set(GENEID_RE.findall(header)))})
with open_source_text(WORK / RNA) as stream:
    for raw in stream:
        line = raw.strip()
        if line.startswith(">"):
            flush()
            header, parts = line[1:], []
        else:
            parts.append(line)
    flush()

fasta = pd.DataFrame(rows)
if fasta.empty:
    raise RuntimeError("No versioned NM_/NR_ accessions parsed from the pinned FASTA")
duplicate_rows = int(fasta.duplicated("accession",keep=False).sum())
conflicting = set(fasta.groupby("accession").sequence.nunique().loc[lambda x:x>1].index)
fasta = fasta.loc[~fasta.accession.isin(conflicting)].drop_duplicates("accession").copy()

ann_rows = []
for accession,item in annotation.items():
    ids = sorted(item["gene_ids"])
    ann_rows.append({"accession":accession,"gene_id_gff":ids[0] if len(ids)==1 else None,
                     "gff_gene_id_count":len(ids),"gff_status":label_from_gff(item),
                     "gff_biotype":";".join(sorted(item["biotypes"])) or None,
                     "has_cds":item["has_cds"]})
df = fasta.merge(pd.DataFrame(ann_rows),on="accession",how="left")
df["gff_status"] = df.gff_status.fillna("unmatched")
df["gene_id"] = df.gene_id_gff
missing_gff_gene = df.gene_id.isna() & df.gff_gene_id_count.fillna(0).eq(0)
df.loc[missing_gff_gene,"gene_id"] = df.loc[missing_gff_gene,"header_gene_ids"].map(
    lambda value:value[0] if len(value)==1 else None)
df["prefix_label"] = (df.prefix == "NM").astype(int)
df["gff_label"] = df.gff_status.map({"coding":1,"noncoding":0})
df["prefix_mismatch"] = df.gff_label.notna() & df.prefix_label.ne(df.gff_label)
df["label_name"] = df.gff_label.map({1:"coding",0:"noncoding"})
audit = {
    "versioned_NM_NR_fasta_records":int(len(rows)),
    "unique_accessions_after_duplicate_collapse":int(len(fasta)),
    "duplicate_accession_rows":duplicate_rows,
    "conflicting_duplicate_accessions_excluded":int(len(conflicting)),
    "gff_matched":int(df.gff_status.ne("unmatched").sum()),
    "gff_rows_total":gff_rows_total,
    "gff_rows_primary_assembled_molecules":gff_rows_primary,
    "gff_rows_outside_primary_assembled_molecules_excluded":gff_rows_outside_primary,
    "primary_assembled_molecule_seqids":len(primary_seqids),
    "gff_unmatched":int(df.gff_status.eq("unmatched").sum()),
    "gff_coding":int(df.gff_status.eq("coding").sum()),
    "gff_explicit_noncoding":int(df.gff_status.eq("noncoding").sum()),
    "gff_ambiguous":int(df.gff_status.eq("ambiguous").sum()),
    "pseudogenes":int(df.gff_status.eq("pseudogene").sum()),
    "prefix_gff_disagreements":int(df.prefix_mismatch.sum()),
    "missing_gene_id_after_header_fallback":int(df.gene_id.isna().sum()),
    "accessions_with_multiple_primary_GeneIDs":int(df.gff_gene_id_count.fillna(0).gt(1).sum()),
    "invalid_sequences":int((~df.valid_bases).sum()),
    "shorter_than_20_nt":int((df.length<MIN_LENGTH).sum()),
}
print(pd.Series(audit).to_string())
print("\nGFF status by accession prefix")
display(pd.crosstab(df.prefix,df.gff_status,margins=True))
print("\nBiotypes among explicit noncoding annotations")
display(df.loc[df.gff_status.eq("noncoding"),"gff_biotype"].fillna("<missing>").value_counts().head(30).to_frame("records"))


## 3. Group related records before splitting

Every transcript for the same GeneID, every exact duplicate sequence, and every sequence assigned to the same MMseqs2 cluster are joined into connected components. After splitting, separate high-sensitivity all-vs-all searches detect any qualifying cross-boundary sequence links and add those links to the components. The notebook repeats grouping and splitting until all three boundaries have zero detected hits or stops after ten repair rounds. The primary threshold is 80% identity and 80% coverage; only the 20 nt malformed-sequence floor is used.


In [ ]:
eligible = df[
    df.gff_status.isin(["coding","noncoding"]) & (~df.prefix_mismatch)
    & df.gene_id.notna() & df.valid_bases & (df.length>=MIN_LENGTH)
].copy()
eligible["label"] = eligible.gff_label.astype(int)
eligible["label_name"] = eligible.label.map({1:"coding",0:"noncoding"})
full_counts = eligible.label_name.value_counts().to_dict()
rng = np.random.default_rng(SEED)
sample = []
for label in [0,1]:
    part = eligible.loc[eligible.label.eq(label)]
    if len(part)>MAX_PER_CLASS:
        part = part.iloc[rng.choice(len(part),MAX_PER_CLASS,replace=False)]
    sample.append(part)
model_df = pd.concat(sample,ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
if model_df.label.nunique()!=2:
    raise RuntimeError("Both GFF labels are required in the model sample")
print("All eligible records:",full_counts)
print("Modeling sample:",model_df.label_name.value_counts().to_dict())

def cluster_sequences(data,identity):
    tag = int(round(identity*100))
    sequence_digest = hashlib.sha256()
    for sequence in data.sequence.astype(str):
        encoded = sequence.encode("ascii")
        sequence_digest.update(len(encoded).to_bytes(8,"big"))
        sequence_digest.update(encoded)
    data_key = sequence_digest.hexdigest()
    cache = WORK / f"mmseqs_cache_{tag}_{int(round(COVERAGE*100))}_{MMSEQS_SHA256[:12]}_{data_key}"
    cache.mkdir(exist_ok=True,parents=True)
    fasta_path = cache / "sample.fna"
    prefix = cache / "clusters"
    tsv = Path(str(prefix)+"_cluster.tsv")
    marker = cache / "cache_manifest.json"
    expected_ids={f"seq_{i}" for i in range(len(data))}
    valid_cache=False
    if tsv.exists() and marker.exists():
        try:
            saved=json.loads(marker.read_text())
            members=set()
            with open(tsv) as stream:
                for line in stream:
                    fields=line.rstrip("\n").split("\t")
                    if len(fields)>=2:
                        members.update(fields[:2])
            valid_cache=(saved.get("input_sha256")==data_key
                         and saved.get("coverage")==COVERAGE
                         and saved.get("mmseqs_archive_sha256")==MMSEQS_SHA256
                         and saved.get("cluster_tsv_sha256")==digest(tsv,"sha256")
                         and members==expected_ids)
        except (OSError,ValueError):
            valid_cache=False
    if not valid_cache:
        with open(fasta_path,"w") as out:
            for i,seq in enumerate(data.sequence):
                out.write(f">seq_{i}\n{seq}\n")
        if tsv.exists(): tsv.unlink()
        command = [str(mmseqs_cmd),"easy-cluster",str(fasta_path),str(prefix),str(cache/"tmp"),
                   "--min-seq-id",str(identity),"-c",str(COVERAGE),"--cov-mode","0",
                   "--threads",str(max(1,min(4,os.cpu_count() or 1)))]
        subprocess.run(command,check=True,capture_output=True,text=True)
    clusters = {}
    with open(tsv) as stream:
        for line in stream:
            fields = line.rstrip("\n").split("\t")
            if len(fields)>=2:
                clusters[fields[1]]=fields[0]
                clusters.setdefault(fields[0],fields[0])
    result = [clusters.get(f"seq_{i}") for i in range(len(data))]
    if any(value is None for value in result):
        raise RuntimeError("MMseqs2 did not assign every sequence")
    marker.write_text(json.dumps({"input_sha256":data_key,"coverage":COVERAGE,
                                  "mmseqs_archive_sha256":MMSEQS_SHA256,
                                  "cluster_tsv_sha256":digest(tsv,"sha256"),
                                  "sequence_count":len(data)},indent=2))
    return result

def component_ids(data,cluster_ids,extra_edges=()):
    parent=list(range(len(data)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]
            i=parent[i]
        return i
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[b]=a
    for column in ["gene_id","sequence"]:
        first={}
        for i,value in enumerate(data[column].astype(str)):
            if value in first: union(i,first[value])
            else: first[value]=i
    first={}
    for i,value in enumerate(cluster_ids):
        if value in first: union(i,first[value])
        else: first[value]=i
    for left_index,right_index in extra_edges:
        union(int(left_index),int(right_index))
    return [f"component_{find(i)}" for i in range(len(data))]

def split_groups(data,group_col,seed=SEED):
    for offset in range(200):
        rs=seed+offset
        outer=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=rs)
        trainval_idx,test_idx=next(outer.split(data,groups=data[group_col]))
        trainval,test=data.iloc[trainval_idx],data.iloc[test_idx]
        inner=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=rs)
        train_idx,val_idx=next(inner.split(trainval,groups=trainval[group_col]))
        train,val=trainval.iloc[train_idx],trainval.iloc[val_idx]
        if all(set(part.label)=={0,1} for part in [train,val,test]):
            return train.copy(),val.copy(),test.copy(),rs
    raise RuntimeError(f"Unable to make three two-class splits for {group_col}")

def verify_split(parts,cluster_col,component_col):
    for a,b in [("train","validation"),("train","test"),("validation","test")]:
        left,right=parts[a],parts[b]
        for col in ["gene_id","sequence",cluster_col,component_col]:
            if set(left[col]) & set(right[col]):
                raise AssertionError(f"Leakage in {col}: {a} vs {b}")
    return True

model_df["cluster_80"] = cluster_sequences(model_df,0.80)

def check_cross_split_similarity(left,right,pair_name,identity=0.80):
    pair_dir=WORK/f"pairwise_{pair_name}"
    pair_dir.mkdir(exist_ok=True,parents=True)
    query_path=pair_dir/"query.fna"
    target_path=pair_dir/"target.fna"
    result_path=pair_dir/"hits.tsv"
    tmp_path=pair_dir/"tmp"
    if tmp_path.exists(): shutil.rmtree(tmp_path)
    query_to_index={}
    target_to_index={}
    with open(query_path,"w") as out:
        for i,(row_index,sequence) in enumerate(zip(right.index,right.sequence.astype(str))):
            query_id=f"query_{i}"
            query_to_index[query_id]=int(row_index)
            out.write(f">{query_id}\n{sequence}\n")
    with open(target_path,"w") as out:
        for i,(row_index,sequence) in enumerate(zip(left.index,left.sequence.astype(str))):
            target_id=f"target_{i}"
            target_to_index[target_id]=int(row_index)
            out.write(f">{target_id}\n{sequence}\n")
    if result_path.exists(): result_path.unlink()
    command=[str(mmseqs_cmd),"easy-search",str(query_path),str(target_path),str(result_path),str(tmp_path),
             "--search-type","3","--min-seq-id",str(identity),"-c",str(COVERAGE),"--cov-mode","0",
             "--alignment-mode","3","-s","7","--max-seqs","1000000","--mask","0","--strand","2",
             "--threads",str(max(1,min(4,os.cpu_count() or 1))),
             "--format-output","query,target,pident,qcov,tcov"]
    subprocess.run(command,check=True,capture_output=True,text=True)
    columns=["query","target","pident","qcov","tcov"]
    try:
        hits=pd.read_csv(result_path,sep="\t",header=None,names=columns)
        if not hits.empty and str(hits.iloc[0]["query"]).lower()=="query":
            hits=hits.iloc[1:].reset_index(drop=True)
        for column in ["pident","qcov","tcov"]:
            hits[column]=pd.to_numeric(hits[column],errors="coerce")
        hits=hits[(hits.pident>=identity*100)&(hits.qcov>=COVERAGE)&(hits.tcov>=COVERAGE)]
    except pd.errors.EmptyDataError:
        hits=pd.DataFrame(columns=columns)
    edges={(min(query_to_index[q],target_to_index[t]),max(query_to_index[q],target_to_index[t]))
           for q,t in zip(hits["query"].astype(str),hits["target"].astype(str))
           if q in query_to_index and t in target_to_index}
    summary={"pair":pair_name,"query_records":len(right),"target_records":len(left),
            "similarity_threshold":identity,"coverage_threshold":COVERAGE,
            "search_sensitivity":7,"strand":"both","real_identity_alignment":True,
            "hits_at_or_above_threshold":int(len(hits)),
            "max_reported_identity_percent":float(hits.pident.max()) if len(hits) else None}
    return summary,edges

pairwise_repair_history=[]
pairwise_similarity_checks=None
pairwise_edges=set()
for repair_iteration in range(1,11):
    model_df["component_80"]=component_ids(model_df,model_df.cluster_80.tolist(),pairwise_edges)
    train,val,test,split_seed=split_groups(model_df,"component_80")
    parts={"train":train,"validation":val,"test":test}
    verify_split(parts,"cluster_80","component_80")
    checks=[]
    edges_this_round=set()
    for left,right,pair_name in [
        (train,val,"train_vs_validation"),
        (train,test,"train_vs_test"),
        (val,test,"validation_vs_test"),
    ]:
        summary,edges=check_cross_split_similarity(
            left,right,f"repair_{repair_iteration}_{pair_name}")
        checks.append(summary)
        edges_this_round.update(edges)
    round_table=pd.DataFrame(checks)
    round_hits=int(round_table.hits_at_or_above_threshold.sum())
    new_edges=edges_this_round-pairwise_edges
    pairwise_repair_history.append({"iteration":repair_iteration,
        "detected_cross_split_hits":round_hits,"new_sequence_links_added":len(new_edges)})
    if round_hits==0:
        pairwise_similarity_checks=round_table
        break
    if not new_edges:
        raise RuntimeError("MMseqs2 still detects cross-split hits but no new grouping links can be added")
    pairwise_edges.update(new_edges)
else:
    raise RuntimeError("MMseqs2 found cross-split matches after ten regrouping rounds; do not train or report results")

print("Cross-split MMseqs2 sequence-search checks on final grouping (all hit counts must be zero; heuristic search, both strands)")
display(pairwise_similarity_checks)
print("Grouping repair history (aggregate counts only)")
display(pd.DataFrame(pairwise_repair_history))
split_table=pd.DataFrame([
    {"split":name,"records":len(part),"genes":part.gene_id.nunique(),
     "components":part.component_80.nunique(),"coding":int(part.label.sum()),
     "noncoding":int((part.label==0).sum())} for name,part in parts.items()
])
display(split_table)
print("PASS: no GeneID, exact sequence, MMseqs2 cluster, connected component, or detected pairwise link crosses a split.")


## 4. Compare baseline and sequence models

Models: a majority-class baseline; an interpretable logistic regression on log length, GC fraction, and normalized 3-mer frequencies; and a character 3–5-mer TF-IDF logistic regression. The TF-IDF regularization value is selected using validation AUPRC only. These CPU-friendly models do not need a GPU.


In [ ]:
def metrics(y,pred,score):
    return {"precision":precision_score(y,pred,zero_division=0),
            "recall":recall_score(y,pred,zero_division=0),
            "f1":f1_score(y,pred,zero_division=0),"auroc":roc_auc_score(y,score),
            "auprc":average_precision_score(y,score),"accuracy":accuracy_score(y,pred),
            "balanced_accuracy":balanced_accuracy_score(y,pred)}

def simple_features(sequences):
    kmers=[a+b+c for a in "ACGT" for b in "ACGT" for c in "ACGT"]
    rows=[]
    for seq in sequences:
        counts=defaultdict(int)
        for i in range(len(seq)-2):
            k=seq[i:i+3]
            if k in kmers: counts[k]+=1
        denom=max(1,sum(counts.values()))
        rows.append([np.log1p(len(seq)),(seq.count("G")+seq.count("C"))/max(1,len(seq)),
                     *[counts[k]/denom for k in kmers]])
    return np.asarray(rows,dtype=np.float32)

Xtr,Xv,Xte=[simple_features(part.sequence) for part in [train,val,test]]
prevalence=float(train.label.mean())
base_val=np.repeat(prevalence,len(val))
base_test=np.repeat(prevalence,len(test))
interpretable=Pipeline([
    ("scale",StandardScaler()),
    ("logistic",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED)),
]).fit(Xtr,train.label)
simple_val=interpretable.predict_proba(Xv)[:,1]

candidates={}
for c_value in [0.25,1.0,4.0]:
    model=Pipeline([
        ("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,
                                  max_features=50000,sublinear_tf=True,dtype=np.float32)),
        ("logistic",LogisticRegression(max_iter=1000,C=c_value,class_weight="balanced",random_state=SEED)),
    ]).fit(train.sequence,train.label)
    score=model.predict_proba(val.sequence)[:,1]
    candidates[c_value]=(average_precision_score(val.label,score),model)
best_c=max(candidates,key=lambda c:candidates[c][0])
best_validation_auprc,selected_val_model=candidates[best_c]
validation_results=pd.DataFrame([
    {"model":"majority baseline",**metrics(val.label,(base_val>=0.5).astype(int),base_val)},
    {"model":"length + GC + 3-mer logistic",**metrics(val.label,interpretable.predict(Xv),simple_val)},
    {"model":f"TF-IDF logistic C={best_c:g}",**metrics(
        val.label,selected_val_model.predict(val.sequence),
        selected_val_model.predict_proba(val.sequence)[:,1])},
])
print("Validation AUPRC by TF-IDF C:",{str(k):round(v[0],4) for k,v in candidates.items()})
display(validation_results.round(4))

trainval=pd.concat([train,val],ignore_index=True)
Xtrainval_simple=simple_features(trainval.sequence)
interpretable.fit(Xtrainval_simple,trainval.label)
base_test=np.repeat(float(trainval.label.mean()),len(test))
final_model=Pipeline([
    ("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,
                              max_features=50000,sublinear_tf=True,dtype=np.float32)),
    ("logistic",LogisticRegression(max_iter=1000,C=float(best_c),class_weight="balanced",random_state=SEED)),
]).fit(trainval.sequence,trainval.label)


## 5. Locked held-out results and error review

The primary test set is evaluated after model selection. Confidence intervals resample whole connected components, so related transcripts do not count as independent observations. Error outputs are aggregate tables without sequences or record-level identifiers.


In [ ]:
def grouped_ci(y,score,groups,iterations=400):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float); groups=np.asarray(groups,dtype=str)
    unique=np.unique(groups)
    by_group={g:np.flatnonzero(groups==g) for g in unique}
    rng=np.random.default_rng(SEED)
    values=[]
    for _ in range(iterations):
        chosen=rng.choice(unique,size=len(unique),replace=True)
        idx=np.concatenate([by_group[g] for g in chosen])
        yy,ss=y[idx],score[idx]
        if len(np.unique(yy))<2: continue
        pp=(ss>=0.5).astype(int)
        values.append([precision_score(yy,pp,zero_division=0),recall_score(yy,pp,zero_division=0),
                       f1_score(yy,pp,zero_division=0),roc_auc_score(yy,ss),average_precision_score(yy,ss)])
    if len(values)<30: return ["unavailable"]*5
    bounds=np.quantile(values,[0.025,0.975],axis=0)
    return [f"[{bounds[0,i]:.3f}, {bounds[1,i]:.3f}]" for i in range(5)]

simple_score=interpretable.predict_proba(Xte)[:,1]
tfidf_score=final_model.predict_proba(test.sequence)[:,1]
model_scores={"majority baseline":base_test,"length + GC + 3-mer logistic":simple_score,
              f"TF-IDF logistic C={best_c:g}":tfidf_score}
test_rows=[]
for name,score in model_scores.items():
    pred=(score>=0.5).astype(int)
    row={"model":name,**metrics(test.label,pred,score),
         "n_test_records":int(len(test)),"n_test_components":int(test.component_80.nunique())}
    for metric,interval in zip(["precision","recall","f1","auroc","auprc"],grouped_ci(test.label,score,test.component_80)):
        row[metric+"_group_bootstrap_95ci"]=interval
    test_rows.append(row)
test_results=pd.DataFrame(test_rows)
display(test_results.round(4))
tfidf_name=f"TF-IDF logistic C={best_c:g}"
tfidf_pred=(tfidf_score>=0.5).astype(int)
print("TF-IDF confusion matrix [rows=true 0/1, columns=predicted 0/1]")
display(pd.DataFrame(confusion_matrix(test.label,tfidf_pred,labels=[0,1]),
                     index=["true noncoding","true coding"],
                     columns=["predicted noncoding","predicted coding"]))

errors=test[["component_80","label","label_name","length","gff_biotype"]].copy()
errors["correct"]=errors.label.to_numpy()==tfidf_pred
errors["gc_fraction"]=test.sequence.map(lambda s:(s.count("G")+s.count("C"))/max(1,len(s))).to_numpy()
error_by_label=errors.groupby("label_name").agg(
    records=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),
    median_length=("length","median"),median_gc=("gc_fraction","median")).reset_index()
error_by_biotype=errors.assign(gff_biotype=errors.gff_biotype.fillna("<missing>")).groupby(
    ["label_name","gff_biotype"]).agg(records=("correct","size"),
    error_rate=("correct",lambda x:1-x.mean()),median_length=("length","median")
).reset_index().query("records >= 5").sort_values("error_rate",ascending=False)
errors["length_bin"]=pd.qcut(errors.length,4,duplicates="drop")
error_by_length=errors.groupby("length_bin",observed=True).agg(
    records=("correct","size"),error_rate=("correct",lambda x:1-x.mean()),
    median_gc=("gc_fraction","median")).reset_index()
print("Errors by label"); display(error_by_label.round(4))
print("Errors by biotype (at least five records)"); display(error_by_biotype.round(4))
print("Errors by held-out length quartile"); display(error_by_length.round(4))


## 6. Similarity-threshold sensitivity and saved outputs

The primary 80% identity and 80% coverage grouping is strengthened iteratively: every qualifying MMseqs2 hit crossing a split is added as a sequence-link between connected groups, then the grouped split and three boundary searches are repeated. Training proceeds only when all three searches find zero qualifying hits; the notebook stops if no new links can be added or ten rounds are insufficient. MMseqs2 is heuristic, so zero detected hits means no qualifying matches under these settings, not proof of no possible similarity. The threshold sweep repeats grouped splits at 70%, 75%, 80%, 85%, 90%, and 95% identity with coverage fixed at 80%. Every threshold split also inherits the primary repair links, keeping the detected 80% cross-split pairs together throughout the sweep. Cluster-overlap sensitivity is descriptive and does not select or change the primary test score.


In [ ]:
threshold_rows=[]
leakage_rows=[]
sweeps={}
for identity in IDENTITIES:
    tag=int(round(identity*100))
    ccol=f"cluster_{tag}"
    gcol=f"component_{tag}"
    model_df[ccol]=cluster_sequences(model_df,identity)
    model_df[gcol]=component_ids(model_df,model_df[ccol].tolist(),pairwise_edges)
    tr,va,te,rs=split_groups(model_df,gcol,split_seed)
    sweep_parts={"train":tr,"validation":va,"test":te}
    verify_split(sweep_parts,ccol,gcol)
    sweeps[identity]=(tr,va,te,rs)

for split_identity,(tr,va,te,rs) in sweeps.items():
    for audit_identity in IDENTITIES:
        col=f"cluster_{int(round(audit_identity*100))}"
        leakage_rows.append({"split_identity":split_identity,"audit_identity":audit_identity,
            "shared_train_test_clusters":len(set(model_df.loc[tr.index,col])&set(model_df.loc[te.index,col])),
            "shared_train_validation_clusters":len(set(model_df.loc[tr.index,col])&set(model_df.loc[va.index,col])),
            "shared_validation_test_clusters":len(set(model_df.loc[va.index,col])&set(model_df.loc[te.index,col]))})
    model=Pipeline([
        ("tfidf",TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,
                                  max_features=50000,sublinear_tf=True,dtype=np.float32)),
        ("logistic",LogisticRegression(max_iter=1000,C=float(best_c),class_weight="balanced",random_state=SEED)),
    ]).fit(pd.concat([tr,va]).sequence,pd.concat([tr,va]).label)
    score=model.predict_proba(te.sequence)[:,1]
    row={"split_identity":split_identity,"coverage":COVERAGE,
         "clusters":int(model_df[f"cluster_{int(round(split_identity*100))}"].nunique()),
         "components":int(model_df[f"component_{int(round(split_identity*100))}"].nunique()),
         "train_records":len(tr),"validation_records":len(va),"test_records":len(te),
         "test_components":int(te[f"component_{int(round(split_identity*100))}"].nunique()),
         **metrics(te.label,(score>=0.5).astype(int),score)}
    own=f"cluster_{int(round(split_identity*100))}"
    row["own_threshold_train_test_cluster_overlap"]=len(set(tr[own])&set(te[own]))
    threshold_rows.append(row)

cluster_overlap_sensitivity=pd.DataFrame(leakage_rows)
threshold_results=pd.DataFrame(threshold_rows).sort_values("split_identity")
assert (threshold_results.own_threshold_train_test_cluster_overlap==0).all()
print("Train/test cluster overlap matrix (diagonal must be zero)")
display(cluster_overlap_sensitivity.pivot(index="split_identity",columns="audit_identity",values="shared_train_test_clusters"))
display(threshold_results.round(4))

validation_results.to_csv(OUTPUT/"validation_model_comparison.csv",index=False)
test_results.to_csv(OUTPUT/"heldout_test_model_comparison.csv",index=False)
error_by_label.to_csv(OUTPUT/"heldout_error_by_label.csv",index=False)
error_by_biotype.to_csv(OUTPUT/"heldout_error_by_biotype.csv",index=False)
error_by_length.to_csv(OUTPUT/"heldout_error_by_length.csv",index=False)
cluster_overlap_sensitivity.to_csv(OUTPUT/"cluster_overlap_sensitivity.csv",index=False)
threshold_results.to_csv(OUTPUT/"identity_threshold_sensitivity.csv",index=False)

manifest={
 "study":"human RefSeq transcript annotated coding-status sequence classification",
 "question":"Sequence-feature prediction of GFF3-annotated CDS versus explicitly noncoding status",
 "scope":{"assembly":ASSEMBLY,"assembly_name":ASSEMBLY_NAME,"annotation_release":RELEASE,
          "known_prefixes":["NM_","NR_"],"predicted_prefixes_excluded":["XM_","XR_"],
          "pseudogenes_excluded":True,"variants_included":False,"FLIP2_included":False,
          "label_rule":"CDS feature versus no CDS plus explicit noncoding GFF3 feature/biotype"},
 "annotation_report_url":REPORT_URL,
 "source":{"release_url":RELEASE_URL,"md5checksums_sha256":digest(checksum_file,"sha256"),
           "files":sources,"mmseqs2":{"version":mmseqs_version,"input_representation":mmseqs_representation,
           "archive_sha256":digest(mmseqs_archive,"sha256") if mmseqs_archive is not None else None,
           "binary_sha256":digest(mmseqs_cmd,"sha256")}},
 "retrieved_utc":datetime.now(timezone.utc).isoformat(),"audit":audit,
 "eligible_by_label":full_counts,"model_sample_by_label":model_df.label_name.value_counts().to_dict(),
 "model_sample_n":int(len(model_df)),"min_length_nt":MIN_LENGTH,
 "split":{"seed":int(split_seed),"primary_identity":0.80,"coverage":COVERAGE,
          "train_records":len(train),"validation_records":len(val),"test_records":len(test),
          "train_components":train.component_80.nunique(),"validation_components":val.component_80.nunique(),
          "test_components":test.component_80.nunique(),"gene_overlap":0,"exact_sequence_overlap":0,
          "similarity_cluster_overlap_at_primary_threshold":0,
          "cross_split_pairwise_search":pairwise_similarity_checks.to_dict("records"),
          "pairwise_repair_history":pairwise_repair_history,
          "pairwise_sequence_links_added":len(pairwise_edges)},
 "models":{"baseline":"majority class","interpretable":"log length, GC, normalized 3-mer logistic regression",
           "sequence":"character 3-5-mer TF-IDF logistic regression",
           "validation_C_values":[0.25,1.0,4.0],"selected_C":float(best_c),
           "selected_validation_auprc":float(best_validation_auprc)},
 "heldout_test_results":test_results.to_dict("records"),
 "errors_by_label":error_by_label.to_dict("records"),
 "thresholds":IDENTITIES,"cluster_overlap_sensitivity":cluster_overlap_sensitivity.to_dict("records"),
 "threshold_results":threshold_results.to_dict("records"),
 "limitations":["One GRCh38.p14 annotation snapshot; this is not an experimental translation assay.",
                "Modeling sample is capped at 3000 eligible transcripts per class, seed 42; transcripts are sampled uniformly and metrics are transcript-weighted.",
                "Gene and sequence grouping is applied to the modeled sample; directly detected MMseqs2 cross-split hits are iteratively added as grouping links until the final three boundary checks return zero. The threshold sweep is descriptive.",
                "Threshold sweep is repeated descriptive test evaluation.",
                "External assembly validation and independent bioinformatics review are pending."]}
manifest_path=OUTPUT/"transcript_coding_status_manifest.json"
manifest_path.write_text(json.dumps(manifest,indent=2,default=str))
print("Aggregate artifacts saved to",OUTPUT.resolve())
print(json.dumps({"release":RELEASE,"audit":audit,"heldout_test_results":manifest["heldout_test_results"],
                  "threshold_rows":len(manifest["threshold_results"]),
                  "output_files":sorted(p.name for p in OUTPUT.iterdir())},indent=2,default=str))


## Interpretation

Read held-out performance beside the majority baseline, class counts, gene/component support, group-bootstrap intervals, and error summaries. Strong performance can reflect sequence composition and annotation conventions; it does not establish translation. The independent bioinformatics reviewer checklist is in the repository, and external review is pending.

### Observed Kaggle run (2026-10-01)

The completed run evaluated the locked 80% identity / 80% coverage split (1,180 test transcripts in 943 connected components). Majority baseline, length + GC + 3-mer logistic regression, and character 3–5-mer TF-IDF logistic regression (C=4) achieved AUPRC 0.4856, 0.7695, and 0.7699; F1 was 0.6537, 0.7338, and 0.7394. The fitted models' connected-component bootstrap intervals overlap, so the small point-estimate difference does not establish a clear winner. The TF-IDF confusion matrix was 406/201 noncoding and 119/454 coding (true negatives/false positives and false negatives/true positives). Noncoding <code>misc_rna</code> had the largest reviewed subgroup error (191 transcripts; 81.7%). The complete audit, error tables, threshold-sensitivity results, and limitations are on the [published findings page](https://minagayid.github.io/flyrank/docs/human-transcript-coding-status/). Rerun the cells above to regenerate these results from the pinned private input bundle.
